# MCMC Sampling Methods for Bayesian Linear Regression
## Comparing Metropolis-Hastings, Gibbs Sampling, and Hamiltonian Monte Carlo

**Authors:** Elad Dagmi & Shaked Mizrahi  
**Course:** Advanced Methods in Machine Learning  
**Date:** August 2026

---

## Table of Contents
1. [Imports & Setup](#imports)
2. [Data Loading & EDA](#eda)
3. [Feature Engineering](#features)
4. [Bayesian Linear Regression Model](#model)
5. [Metropolis-Hastings (MH)](#mh)
6. [Gibbs Sampling](#gibbs)
7. [Hamiltonian Monte Carlo (HMC)](#hmc)
8. [Running the Samplers](#run)
9. [Diagnostics & Convergence](#diagnostics)
10. [Results & Comparison](#results)
11. [Conclusions](#conclusions)

## 1. Imports & Setup <a id='imports'></a>

In [ ]:
import os
import glob
import time
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.special import gammaln

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})
np.random.seed(42)

PROJECT_ROOT = os.getcwd()
if os.path.basename(PROJECT_ROOT) == 'notebooks':
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
DATA_ROOT = os.path.join(PROJECT_ROOT, 'data')
TRACE_DATA_DIR = os.path.join(DATA_ROOT, 'fastStorage', '2013-8')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'results', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Figures are written to results/figures/ so the notebook can run from notebooks/.
def figure_path(figure_filename):
    return os.path.join(FIGURES_DIR, figure_filename)

print('Setup complete.')

## 2. Data Loading & EDA <a id='eda'></a>

We use the **Bitbrains Datacenter Traces** dataset containing performance metrics of 1,250 VMs from a real datacenter, sampled every 5 minutes over ~30 days.

In [ ]:
DATA_DIR = TRACE_DATA_DIR
ZIP_PATH = os.path.join(DATA_ROOT, 'fastStorage.zip')

if not os.path.isdir(DATA_DIR) and os.path.isfile(ZIP_PATH):
    print('Extracting zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_ROOT)
    print('Extracted.')

COLUMNS = [
    'Timestamp', 'CPU_Cores', 'CPU_Capacity_MHz', 'CPU_Usage_MHz',
    'CPU_Usage_Pct', 'Mem_Provisioned_KB', 'Mem_Usage_KB',
    'Disk_Read_KBps', 'Disk_Write_KBps', 'Net_Recv_KBps', 'Net_Trans_KBps'
]

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print(f'Found {len(csv_files)} VM trace files.')

sample_df = pd.read_csv(csv_files[0], sep=';\t', header=0, engine='python')
sample_df.columns = COLUMNS
print(f'\nSample VM shape: {sample_df.shape}')
sample_df.head()

In [ ]:
NUM_VMS = 50

dfs = []
for f in csv_files[:NUM_VMS]:
    vm_df = pd.read_csv(f, sep=';\t', header=0, engine='python')
    vm_df.columns = COLUMNS
    vm_df['VM_ID'] = os.path.basename(f).replace('.csv', '')
    dfs.append(vm_df)

data = pd.concat(dfs, ignore_index=True)
data['Datetime'] = pd.to_datetime(data['Timestamp'], unit='s')
print(f'Combined dataset: {data.shape[0]:,} rows from {NUM_VMS} VMs')
print(f'Date range: {data["Datetime"].min()} to {data["Datetime"].max()}')
data.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

plot_cols = ['CPU_Usage_Pct', 'Mem_Usage_KB', 'Disk_Read_KBps',
             'Disk_Write_KBps', 'Net_Recv_KBps', 'Net_Trans_KBps']
plot_names = ['CPU Usage (%)', 'Memory Usage (KB)', 'Disk Read (KB/s)',
              'Disk Write (KB/s)', 'Net Received (KB/s)', 'Net Transmitted (KB/s)']

for ax, col, name in zip(axes.flat, plot_cols, plot_names):
    ax.hist(data[col].dropna(), bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_title(name)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(figure_path('eda_histograms.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample_vm = data[data['VM_ID'] == data['VM_ID'].unique()[0]].copy()
sample_vm = sample_vm.sort_values('Datetime').reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'Time Series for VM: {sample_vm["VM_ID"].iloc[0]}', fontsize=14, fontweight='bold')

axes[0].plot(sample_vm['Datetime'], sample_vm['CPU_Usage_Pct'], linewidth=0.5, color='steelblue')
axes[0].set_ylabel('CPU Usage (%)')

axes[1].plot(sample_vm['Datetime'], sample_vm['Mem_Usage_KB'] / 1024, linewidth=0.5, color='coral')
axes[1].set_ylabel('Memory Usage (MB)')

axes[2].plot(sample_vm['Datetime'], sample_vm['Disk_Read_KBps'], linewidth=0.5, color='seagreen')
axes[2].set_ylabel('Disk Read (KB/s)')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.savefig(figure_path('eda_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = ['CPU_Usage_Pct', 'CPU_Cores', 'CPU_Capacity_MHz', 'CPU_Usage_MHz',
                'Mem_Provisioned_KB', 'Mem_Usage_KB',
                'Disk_Read_KBps', 'Disk_Write_KBps', 'Net_Recv_KBps', 'Net_Trans_KBps']

corr_matrix = data[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
short_names = ['CPU%', 'Cores', 'CPUCap', 'CPUMHz', 'MemProv', 'MemUse', 'DskR', 'DskW', 'NetR', 'NetT']
ax.set_xticklabels(short_names, rotation=45, ha='right')
ax.set_yticklabels(short_names)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Feature Correlation Matrix (All Original Variables, Before Feature Selection)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(figure_path('eda_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering <a id='features'></a>

We engineer features for predicting CPU usage (%):
- **Raw features:** Memory Usage, Disk Read/Write, Network Recv/Trans (CPU Usage MHz excluded to avoid data leakage)
- **Lag features:** Previous time-step CPU usage values (t-1, t-2, t-3)
- **Rolling statistics:** Rolling mean and std over 6-step window (30 min), computed from past values only
- **Normalization:** Z-score standardization

**Note:** CPU Usage (MHz) was excluded from the predictor set because it directly encodes the same quantity as the target variable CPU Usage (%) at the same timestamp (MHz / Capacity = %), which would constitute data leakage.

In [ ]:
def engineer_features(vm_data):
    """Create lag and rolling features for a single VM."""
    df = vm_data.sort_values('Datetime').copy()
    target_col = 'CPU_Usage_Pct'
    # CPU_Usage_MHz excluded to avoid data leakage (MHz / Capacity = %)
    feature_cols = ['Mem_Usage_KB', 'Disk_Read_KBps',
                    'Disk_Write_KBps', 'Net_Recv_KBps', 'Net_Trans_KBps']
    for lag in [1, 2, 3]:
        df[f'CPU_lag_{lag}'] = df[target_col].shift(lag)
    window = 6
    df['CPU_rolling_mean'] = df[target_col].shift(1).rolling(window=window).mean()
    df['CPU_rolling_std'] = df[target_col].shift(1).rolling(window=window).std()
    df = df.dropna().reset_index(drop=True)
    predictors = feature_cols + [f'CPU_lag_{i}' for i in [1, 2, 3]] + ['CPU_rolling_mean', 'CPU_rolling_std']
    return df, predictors, target_col

vm_ids = data['VM_ID'].unique()
all_features = []
for vm_id in vm_ids:
    vm_data = data[data['VM_ID'] == vm_id]
    if len(vm_data) < 50:
        continue
    feat_df, predictor_names, target_name = engineer_features(vm_data)
    all_features.append(feat_df)

features_df = pd.concat(all_features, ignore_index=True)
print(f'Engineered dataset: {features_df.shape[0]:,} rows, {len(predictor_names)} predictors')
print(f'Predictors: {predictor_names}')

In [ ]:
MAX_SAMPLES = 5000
# Sort chronologically and take earliest observations to preserve temporal order
features_df = features_df.sort_values('Datetime').reset_index(drop=True)
if len(features_df) > MAX_SAMPLES:
    features_df = features_df.iloc[:MAX_SAMPLES].reset_index(drop=True)

X_raw = features_df[predictor_names].values
y_raw = features_df[target_name].values

X_mean, X_std = X_raw.mean(axis=0), X_raw.std(axis=0)
X_std[X_std == 0] = 1.0
X_scaled = (X_raw - X_mean) / X_std

y_mean, y_std = y_raw.mean(), y_raw.std()
if y_std == 0:
    y_std = 1.0
y_scaled = (y_raw - y_mean) / y_std

X = np.column_stack([np.ones(len(X_scaled)), X_scaled])
y = y_scaled

n, p = X.shape
split = int(0.7 * n)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features (incl. intercept)')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Temporal split: train uses earlier observations, test uses later ones')

## 4. Bayesian Linear Regression Model <a id='model'></a>

### Model Specification

**Likelihood:**
$$y \mid X, \beta, \sigma^2 \sim \mathcal{N}(X\beta,\; \sigma^2 I)$$

**Priors (conjugate):**
$$\beta \sim \mathcal{N}(0,\; \tau^2 I)$$
$$\sigma^2 \sim \text{Inverse-Gamma}(a_0,\; b_0)$$

**Posterior:**
$$p(\beta, \sigma^2 \mid y, X) \propto p(y \mid X, \beta, \sigma^2)\; p(\beta)\; p(\sigma^2)$$

In [ ]:
TAU2 = 10.0
A0 = 2.0
B0 = 1.0

def log_likelihood(beta, sigma2, X, y):
    """Compute log p(y | X, beta, sigma2)."""
    n = len(y)
    residuals = y - X @ beta
    return -0.5 * n * np.log(2 * np.pi * sigma2) - 0.5 * np.sum(residuals**2) / sigma2

def log_prior_beta(beta, tau2=TAU2):
    """Compute log p(beta) under N(0, tau2*I)."""
    p = len(beta)
    return -0.5 * p * np.log(2 * np.pi * tau2) - 0.5 * np.sum(beta**2) / tau2

def log_prior_sigma2(sigma2, a0=A0, b0=B0):
    """Compute log p(sigma2) under Inverse-Gamma(a0, b0)."""
    if sigma2 <= 0:
        return -np.inf
    return a0 * np.log(b0) - gammaln(a0) - (a0 + 1) * np.log(sigma2) - b0 / sigma2

def log_posterior(beta, sigma2, X, y):
    """Compute log p(beta, sigma2 | y, X) up to a constant."""
    lp_sigma = log_prior_sigma2(sigma2)
    if np.isinf(lp_sigma):
        return -np.inf
    return log_likelihood(beta, sigma2, X, y) + log_prior_beta(beta) + lp_sigma

print(f'Model: {p} parameters (beta) + 1 (sigma^2)')
print(f'Priors: beta ~ N(0, {TAU2}I), sigma^2 ~ IG({A0}, {B0})')

## 5. Metropolis-Hastings (MH) <a id='mh'></a>

The MH algorithm proposes new parameters from a symmetric random-walk proposal:
$$\theta' = \theta + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \Sigma_{\text{prop}})$$

Acceptance ratio (for symmetric proposal):
$$\alpha = \min\left(1,\; \frac{\pi(\theta')}{\pi(\theta)}\right)$$

In [ ]:
def metropolis_hastings(X, y, n_samples, burn_in, step_beta=0.01, step_sigma2=0.05):
    """
    Metropolis-Hastings sampler for Bayesian linear regression.
    Uses symmetric random-walk proposals for beta and log(sigma2).
    """
    n, p = X.shape
    beta = np.zeros(p)
    sigma2 = 1.0
    log_sigma2 = np.log(sigma2)
    current_lp = log_posterior(beta, sigma2, X, y)
    total = n_samples + burn_in
    beta_samples = np.zeros((n_samples, p))
    sigma2_samples = np.zeros(n_samples)
    accepted = 0
    for i in range(total):
        beta_prop = beta + np.random.normal(0, step_beta, size=p)
        log_sigma2_prop = log_sigma2 + np.random.normal(0, step_sigma2)
        sigma2_prop = np.exp(log_sigma2_prop)
        proposed_lp = log_posterior(beta_prop, sigma2_prop, X, y)
        log_jacobian = log_sigma2_prop - log_sigma2
        log_alpha = proposed_lp - current_lp + log_jacobian
        if np.log(np.random.uniform()) < log_alpha:
            beta = beta_prop
            sigma2 = sigma2_prop
            log_sigma2 = log_sigma2_prop
            current_lp = proposed_lp
            if i >= burn_in:
                accepted += 1
        if i >= burn_in:
            beta_samples[i - burn_in] = beta
            sigma2_samples[i - burn_in] = sigma2
    acceptance_rate = accepted / n_samples
    return beta_samples, sigma2_samples, acceptance_rate

print('Metropolis-Hastings sampler defined.')

## 6. Gibbs Sampling <a id='gibbs'></a>

Gibbs sampling draws each parameter from its **full conditional distribution**:

$$\beta \mid \sigma^2, y, X \sim \mathcal{N}(\mu_\beta,\; \Sigma_\beta)$$
$$\Sigma_\beta = \left(\frac{X^T X}{\sigma^2} + \frac{I}{\tau^2}\right)^{-1}, \quad \mu_\beta = \Sigma_\beta \frac{X^T y}{\sigma^2}$$

$$\sigma^2 \mid \beta, y, X \sim \text{Inverse-Gamma}(a_n,\; b_n)$$
$$a_n = a_0 + \frac{n}{2}, \quad b_n = b_0 + \frac{\|y - X\beta\|^2}{2}$$

The acceptance rate is **100%** since we sample from the exact conditionals. This is a structural property of the algorithm when conjugate priors are available, and should not be interpreted as direct evidence of superior sampling quality. The samples are still generated sequentially by a Markov chain and are not theoretically independent.

In [ ]:
def gibbs_sampling(X, y, n_samples, burn_in, tau2=TAU2, a0=A0, b0=B0):
    """
    Gibbs sampler for Bayesian linear regression with conjugate priors.
    Alternates between sampling beta|sigma2 and sigma2|beta.
    """
    n, p = X.shape
    XtX = X.T @ X
    Xty = X.T @ y
    beta = np.zeros(p)
    sigma2 = 1.0
    total = n_samples + burn_in
    beta_samples = np.zeros((n_samples, p))
    sigma2_samples = np.zeros(n_samples)
    for i in range(total):
        precision_beta = XtX / sigma2 + np.eye(p) / tau2
        cov_beta = np.linalg.inv(precision_beta)
        mean_beta = cov_beta @ (Xty / sigma2)
        beta = np.random.multivariate_normal(mean_beta, cov_beta)
        residuals = y - X @ beta
        a_n = a0 + n / 2.0
        b_n = b0 + 0.5 * np.sum(residuals**2)
        sigma2 = 1.0 / np.random.gamma(a_n, 1.0 / b_n)
        if i >= burn_in:
            beta_samples[i - burn_in] = beta
            sigma2_samples[i - burn_in] = sigma2
    return beta_samples, sigma2_samples, 1.0

print('Gibbs sampler defined.')

## 7. Hamiltonian Monte Carlo (HMC) <a id='hmc'></a>

HMC uses the gradient of the log-posterior to make informed proposals via Hamiltonian dynamics.

**Hamiltonian:**
$$H(q, p) = U(q) + K(p) = -\log \pi(q) + \frac{1}{2} p^T p$$

where $U(q) = -\log \pi(q)$ is the potential energy and $K(p)$ is the kinetic energy.

**Leapfrog integrator** (repeated $L$ times with step size $\varepsilon$):
1. **Half-step momentum:**  $\; p \;\leftarrow\; p \;-\; \frac{\varepsilon}{2} \cdot \nabla U(q)$
2. **Full-step position:**  $\; q \;\leftarrow\; q \;+\; \varepsilon \cdot p$
3. **Half-step momentum:**  $\; p \;\leftarrow\; p \;-\; \frac{\varepsilon}{2} \cdot \nabla U(q)$

After $L$ leapfrog steps, apply Metropolis accept/reject using $\Delta H = H(q', p') - H(q, p)$.

In [ ]:
def grad_log_posterior(theta, X, y, tau2=TAU2, a0=A0, b0=B0):
    """
    Gradient of log-posterior w.r.t. theta = [beta, log_sigma2].
    We parameterize sigma2 = exp(log_sigma2) for unconstrained sampling.
    """
    p_features = X.shape[1]
    beta = theta[:p_features]
    log_sigma2 = theta[p_features]
    sigma2 = np.exp(log_sigma2)
    n = len(y)
    residuals = y - X @ beta
    grad_beta = (X.T @ residuals) / sigma2 - beta / tau2
    grad_log_sigma2 = (
        -0.5 * n
        + 0.5 * np.sum(residuals**2) / sigma2
        - (a0 + 1)
        + b0 / sigma2
        + 1  # Jacobian correction
    )
    return np.concatenate([grad_beta, [grad_log_sigma2]])


def log_posterior_hmc(theta, X, y, tau2=TAU2, a0=A0, b0=B0):
    """Log-posterior in terms of theta = [beta, log_sigma2]."""
    p_features = X.shape[1]
    beta = theta[:p_features]
    log_sigma2 = theta[p_features]
    sigma2 = np.exp(log_sigma2)
    lp = log_posterior(beta, sigma2, X, y)
    lp += log_sigma2  # Jacobian for log-transform
    return lp


def hmc_sampler(X, y, n_samples, burn_in, step_size=0.001, n_leapfrog=20):
    """
    Hamiltonian Monte Carlo sampler for Bayesian linear regression.
    Samples in unconstrained space: theta = [beta, log(sigma2)].
    """
    n, p_features = X.shape
    d = p_features + 1
    theta = np.zeros(d)
    current_lp = log_posterior_hmc(theta, X, y)
    total = n_samples + burn_in
    beta_samples = np.zeros((n_samples, p_features))
    sigma2_samples = np.zeros(n_samples)
    accepted = 0
    for i in range(total):
        momentum = np.random.normal(0, 1, size=d)
        theta_prop = theta.copy()
        momentum_prop = momentum.copy()
        grad = grad_log_posterior(theta_prop, X, y)
        momentum_prop += 0.5 * step_size * grad
        for _ in range(n_leapfrog):
            theta_prop += step_size * momentum_prop
            grad = grad_log_posterior(theta_prop, X, y)
            momentum_prop += step_size * grad
        momentum_prop -= 0.5 * step_size * grad
        momentum_prop = -momentum_prop
        proposed_lp = log_posterior_hmc(theta_prop, X, y)
        current_ke = 0.5 * np.sum(momentum**2)
        proposed_ke = 0.5 * np.sum(momentum_prop**2)
        log_alpha = proposed_lp - current_lp - proposed_ke + current_ke
        if np.log(np.random.uniform()) < log_alpha:
            theta = theta_prop
            current_lp = proposed_lp
            if i >= burn_in:
                accepted += 1
        if i >= burn_in:
            beta_samples[i - burn_in] = theta[:p_features]
            sigma2_samples[i - burn_in] = np.exp(theta[p_features])
    acceptance_rate = accepted / n_samples
    return beta_samples, sigma2_samples, acceptance_rate

print('HMC sampler defined.')

## 8. Running the Samplers <a id='run'></a>

In [ ]:
N_SAMPLES = 10000
BURN_IN = 2000
N_CHAINS = 3

results = {}

print('=' * 70)
print('Running Metropolis-Hastings...')
print('=' * 70)
mh_chains_beta, mh_chains_sigma2 = [], []
start = time.time()
for chain in range(N_CHAINS):
    np.random.seed(42 + chain)
    b, s, ar = metropolis_hastings(X_train, y_train, N_SAMPLES, BURN_IN,
                                   step_beta=0.001, step_sigma2=0.05)
    mh_chains_beta.append(b)
    mh_chains_sigma2.append(s)
    print(f'  Chain {chain+1}: acceptance rate = {ar:.3f}')
mh_time = time.time() - start
results['MH'] = {
    'beta_chains': mh_chains_beta,
    'sigma2_chains': mh_chains_sigma2,
    'acceptance_rate': ar,
    'time': mh_time
}
print(f'  Total time: {mh_time:.2f}s\n')

print('=' * 70)
print('Running Gibbs Sampling...')
print('=' * 70)
gibbs_chains_beta, gibbs_chains_sigma2 = [], []
start = time.time()
for chain in range(N_CHAINS):
    np.random.seed(42 + chain)
    b, s, ar = gibbs_sampling(X_train, y_train, N_SAMPLES, BURN_IN)
    gibbs_chains_beta.append(b)
    gibbs_chains_sigma2.append(s)
    print(f'  Chain {chain+1}: acceptance rate = {ar:.3f}')
gibbs_time = time.time() - start
results['Gibbs'] = {
    'beta_chains': gibbs_chains_beta,
    'sigma2_chains': gibbs_chains_sigma2,
    'acceptance_rate': 1.0,
    'time': gibbs_time
}
print(f'  Total time: {gibbs_time:.2f}s\n')

print('=' * 70)
print('Running Hamiltonian Monte Carlo...')
print('=' * 70)
hmc_chains_beta, hmc_chains_sigma2 = [], []
start = time.time()
for chain in range(N_CHAINS):
    np.random.seed(42 + chain)
    b, s, ar = hmc_sampler(X_train, y_train, N_SAMPLES, BURN_IN,
                           step_size=0.002, n_leapfrog=15)
    hmc_chains_beta.append(b)
    hmc_chains_sigma2.append(s)
    print(f'  Chain {chain+1}: acceptance rate = {ar:.3f}')
hmc_time = time.time() - start
results['HMC'] = {
    'beta_chains': hmc_chains_beta,
    'sigma2_chains': hmc_chains_sigma2,
    'acceptance_rate': ar,
    'time': hmc_time
}
print(f'  Total time: {hmc_time:.2f}s')

## 9. Diagnostics & Convergence <a id='diagnostics'></a>

In [ ]:
def effective_sample_size(chain):
    """Compute ESS using autocorrelation method."""
    n = len(chain)
    chain_centered = chain - np.mean(chain)
    variance = np.var(chain_centered)
    if variance == 0:
        return 0.0
    acf = np.correlate(chain_centered, chain_centered, mode='full')[n-1:]
    acf = acf / (variance * n)
    tau = 1.0
    for k in range(1, n // 2):
        if acf[k] < 0.05:
            break
        tau += 2.0 * acf[k]
    return n / tau


def gelman_rubin(chains):
    """Compute Gelman-Rubin R-hat statistic for convergence assessment."""
    m = len(chains)
    n = len(chains[0])
    chain_means = np.array([np.mean(c) for c in chains])
    overall_mean = np.mean(chain_means)
    between_chain_var = n / (m - 1) * np.sum((chain_means - overall_mean)**2)
    within_chain_var = np.mean([np.var(c, ddof=1) for c in chains])
    var_hat = (1 - 1/n) * within_chain_var + (1/n) * between_chain_var
    r_hat = np.sqrt(var_hat / within_chain_var) if within_chain_var > 0 else np.inf
    return r_hat

print('Diagnostic functions defined.')

In [ ]:
param_indices = [0, 1, min(3, p-1)]
param_labels = ['Intercept', 'beta_1', f'beta_{param_indices[2]}']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('Trace Plots (3 chains per method)', fontsize=16, fontweight='bold')
method_names = ['MH', 'Gibbs', 'HMC']
colors = ['steelblue', 'coral', 'seagreen']

for col_idx, method in enumerate(method_names):
    chains = results[method]['beta_chains']
    for row_idx, (pidx, plabel) in enumerate(zip(param_indices, param_labels)):
        ax = axes[row_idx, col_idx]
        for chain_idx in range(N_CHAINS):
            ax.plot(chains[chain_idx][:, pidx], alpha=0.6, linewidth=0.3)
        if row_idx == 0:
            ax.set_title(method, fontsize=14, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(plabel)
        if row_idx == 2:
            ax.set_xlabel('Iteration')

plt.tight_layout()
plt.savefig(figure_path('trace_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Trace Plots for sigma^2', fontsize=14, fontweight='bold')

for col_idx, method in enumerate(method_names):
    ax = axes[col_idx]
    chains = results[method]['sigma2_chains']
    for chain_idx in range(N_CHAINS):
        ax.plot(chains[chain_idx], alpha=0.6, linewidth=0.3)
    ax.set_title(method)
    ax.set_xlabel('Iteration')
    if col_idx == 0:
        ax.set_ylabel('sigma^2')

plt.tight_layout()
plt.savefig(figure_path('trace_sigma2.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(len(param_indices) + 1, 3, figsize=(16, 14))
fig.suptitle('Posterior Distributions', fontsize=16, fontweight='bold')

for col_idx, method in enumerate(method_names):
    beta_combined = np.vstack(results[method]['beta_chains'])
    sigma2_combined = np.concatenate(results[method]['sigma2_chains'])
    for row_idx, (pidx, plabel) in enumerate(zip(param_indices, param_labels)):
        ax = axes[row_idx, col_idx]
        ax.hist(beta_combined[:, pidx], bins=60, density=True, alpha=0.7, color=colors[col_idx])
        ax.axvline(np.mean(beta_combined[:, pidx]), color='red', linestyle='--', linewidth=1.5)
        if row_idx == 0:
            ax.set_title(method, fontsize=13, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(plabel)
    ax = axes[-1, col_idx]
    ax.hist(sigma2_combined, bins=60, density=True, alpha=0.7, color=colors[col_idx])
    ax.axvline(np.mean(sigma2_combined), color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Value')
    if col_idx == 0:
        ax.set_ylabel('sigma^2')

plt.tight_layout()
plt.savefig(figure_path('posterior_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 80)
print(f'{"Parameter":<15} {"Method":<10} {"R-hat":<10} {"ESS":<12} {"Mean":<12} {"Std":<10}')
print('=' * 80)

for method in method_names:
    beta_chains = results[method]['beta_chains']
    sigma2_chains = results[method]['sigma2_chains']
    for pidx, plabel in zip(param_indices, param_labels):
        param_chains = [c[:, pidx] for c in beta_chains]
        rhat = gelman_rubin(param_chains)
        ess = np.mean([effective_sample_size(c) for c in param_chains])
        combined = np.concatenate(param_chains)
        print(f'{plabel:<15} {method:<10} {rhat:<10.4f} {ess:<12.1f} {np.mean(combined):<12.6f} {np.std(combined):<10.6f}')
    rhat_s = gelman_rubin(sigma2_chains)
    ess_s = np.mean([effective_sample_size(c) for c in sigma2_chains])
    combined_s = np.concatenate(sigma2_chains)
    print(f'{"sigma^2":<15} {method:<10} {rhat_s:<10.4f} {ess_s:<12.1f} {np.mean(combined_s):<12.6f} {np.std(combined_s):<10.6f}')
    print('-' * 80)

## 10. Results & Comparison <a id='results'></a>

In [ ]:
def compute_predictions(beta_samples, sigma2_samples, X_test, y_test):
    """Compute RMSE and calibration metrics."""
    n_test = len(y_test)
    n_post = len(beta_samples)
    y_pred_samples = np.zeros((n_post, n_test))
    for i in range(n_post):
        y_pred_samples[i] = X_test @ beta_samples[i]
    y_pred_mean = np.mean(y_pred_samples, axis=0)
    rmse = np.sqrt(np.mean((y_test - y_pred_mean)**2))
    y_pred_std = np.std(y_pred_samples, axis=0)
    avg_sigma = np.sqrt(np.mean(sigma2_samples))
    total_std = np.sqrt(y_pred_std**2 + avg_sigma**2)
    lower_95 = y_pred_mean - 1.96 * total_std
    upper_95 = y_pred_mean + 1.96 * total_std
    coverage_95 = np.mean((y_test >= lower_95) & (y_test <= upper_95))
    lower_50 = y_pred_mean - 0.6745 * total_std
    upper_50 = y_pred_mean + 0.6745 * total_std
    coverage_50 = np.mean((y_test >= lower_50) & (y_test <= upper_50))
    return {
        'rmse': rmse,
        'coverage_95': coverage_95,
        'coverage_50': coverage_50,
        'y_pred_mean': y_pred_mean,
        'lower_95': lower_95,
        'upper_95': upper_95
    }

print('Prediction functions defined.')

In [ ]:
for method in method_names:
    beta_combined = np.vstack(results[method]['beta_chains'])
    sigma2_combined = np.concatenate(results[method]['sigma2_chains'])
    preds = compute_predictions(beta_combined, sigma2_combined, X_test, y_test)
    results[method]['predictions'] = preds
    all_beta_chains = results[method]['beta_chains']
    all_sigma2_chains = results[method]['sigma2_chains']
    ess_list = []
    for pidx in range(p):
        for c in all_beta_chains:
            ess_list.append(effective_sample_size(c[:, pidx]))
    for c in all_sigma2_chains:
        ess_list.append(effective_sample_size(c))
    results[method]['avg_ess'] = np.mean(ess_list)
    results[method]['ess_per_sec'] = results[method]['avg_ess'] / results[method]['time']

# OLS baseline for comparison
beta_ols = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
y_pred_ols = X_test @ beta_ols
ols_rmse = np.sqrt(np.mean((y_test - y_pred_ols) ** 2))
print(f'OLS baseline RMSE: {ols_rmse:.4f}')
print('All results computed.')

In [ ]:
print('\n' + '=' * 90)
print('COMPARISON TABLE')
print('=' * 90)
header = f'{"Metric":<25} {"OLS":<20} {"MH":<20} {"Gibbs":<20} {"HMC":<20}'
print(header)
print('-' * 110)

rows = [
    ('Acceptance Rate', ['\u2014'] + [f"{results[m]['acceptance_rate']:.3f}" for m in method_names]),
    ('Avg ESS', ['\u2014'] + [f"{results[m]['avg_ess']:.1f}" for m in method_names]),
    ('Runtime (s)', ['\u2014'] + [f"{results[m]['time']:.2f}" for m in method_names]),
    ('ESS / second', ['\u2014'] + [f"{results[m]['ess_per_sec']:.1f}" for m in method_names]),
    ('RMSE', [f"{ols_rmse:.4f}"] + [f"{results[m]['predictions']['rmse']:.4f}" for m in method_names]),
    ('95% Coverage', ['\u2014'] + [f"{results[m]['predictions']['coverage_95']:.3f}" for m in method_names]),
    ('50% Coverage', ['\u2014'] + [f"{results[m]['predictions']['coverage_50']:.3f}" for m in method_names]),
]

for metric, values in rows:
    print(f'{metric:<25} {values[0]:<20} {values[1]:<20} {values[2]:<20} {values[3]:<20}')

print('=' * 110)
print('\nNote: OLS provides a frequentist baseline. The Bayesian methods achieve')
print('comparable RMSE but additionally provide uncertainty quantification')
print('through posterior predictive intervals, which OLS does not.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Method Comparison', fontsize=16, fontweight='bold')
bar_colors = ['steelblue', 'coral', 'seagreen']

metrics = [
    ('Acceptance Rate', [results[m]['acceptance_rate'] for m in method_names]),
    ('Average ESS', [results[m]['avg_ess'] for m in method_names]),
    ('Runtime (s)', [results[m]['time'] for m in method_names]),
    ('ESS / second', [results[m]['ess_per_sec'] for m in method_names]),
    ('RMSE', [results[m]['predictions']['rmse'] for m in method_names]),
    ('95% Coverage', [results[m]['predictions']['coverage_95'] for m in method_names]),
]

for ax, (metric_name, values) in zip(axes.flat, metrics):
    bars = ax.bar(method_names, values, color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric_name, fontsize=13)
    ax.set_ylabel(metric_name)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(figure_path('comparison_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Predictions vs Actual (Test Set)', fontsize=16, fontweight='bold')
plot_n = min(200, len(y_test))

for ax, method, color in zip(axes, method_names, bar_colors):
    preds = results[method]['predictions']
    ax.plot(range(plot_n), y_test[:plot_n], 'k-', linewidth=1, label='Actual', alpha=0.8)
    ax.plot(range(plot_n), preds['y_pred_mean'][:plot_n], color=color, linewidth=1,
            label='Predicted', alpha=0.8)
    ax.fill_between(range(plot_n), preds['lower_95'][:plot_n], preds['upper_95'][:plot_n],
                    color=color, alpha=0.15, label='95% CI')
    ax.set_title(f'{method} (RMSE={preds["rmse"]:.4f})', fontsize=13)
    ax.legend(loc='upper right')
    ax.set_ylabel('CPU Usage (scaled)')

axes[-1].set_xlabel('Test Sample Index')
plt.tight_layout()
plt.savefig(figure_path('predictions_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Autocorrelation of beta_0 (Intercept) - Chain 1', fontsize=14, fontweight='bold')
max_lag = 100

for ax, method, color in zip(axes, method_names, bar_colors):
    chain = results[method]['beta_chains'][0][:, 0]
    chain_centered = chain - np.mean(chain)
    acf_full = np.correlate(chain_centered, chain_centered, mode='full')
    acf_full = acf_full[len(chain_centered)-1:]
    acf_full = acf_full / acf_full[0]
    ax.bar(range(max_lag), acf_full[:max_lag], color=color, alpha=0.7)
    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=1)
    ax.axhline(y=-0.05, color='red', linestyle='--', linewidth=1)
    ax.set_title(method)
    ax.set_xlabel('Lag')
    if ax == axes[0]:
        ax.set_ylabel('ACF')

plt.tight_layout()
plt.savefig(figure_path('autocorrelation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Running R-hat Convergence (beta_0)', fontsize=14, fontweight='bold')

checkpoints = np.arange(200, N_SAMPLES + 1, 200)

for ax, method, color in zip(axes, method_names, bar_colors):
    rhats = []
    for cp in checkpoints:
        chains_cp = [c[:cp, 0] for c in results[method]['beta_chains']]
        rhats.append(gelman_rubin(chains_cp))
    ax.plot(checkpoints, rhats, color=color, linewidth=2)
    ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1)
    ax.axhline(y=1.1, color='red', linestyle=':', linewidth=1, label='R-hat = 1.1 threshold')
    ax.set_title(method)
    ax.set_xlabel('Iteration')
    if ax == axes[0]:
        ax.set_ylabel('R-hat')
    ax.legend()
    ax.set_ylim(0.95, max(1.5, max(rhats) + 0.1))

plt.tight_layout()
plt.savefig(figure_path('rhat_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Hyperparameter Sensitivity Analysis')
print('=' * 70)
print('\n--- MH: Step Size Sensitivity ---')
mh_step_sizes = [0.001, 0.005, 0.01, 0.05]
for step in mh_step_sizes:
    np.random.seed(42)
    _, _, ar = metropolis_hastings(X_train, y_train, 3000, 500, step_beta=step, step_sigma2=0.05)
    print(f'  step_beta={step:.3f}: acceptance_rate={ar:.3f}')

print('\n--- HMC: Leapfrog Steps Sensitivity ---')
hmc_leapfrog_values = [5, 10, 15, 25]
for lf in hmc_leapfrog_values:
    np.random.seed(42)
    _, _, ar = hmc_sampler(X_train, y_train, 3000, 500, step_size=0.002, n_leapfrog=lf)
    print(f'  n_leapfrog={lf}: acceptance_rate={ar:.3f}')

## 11. Conclusions <a id='conclusions'></a>

This project focuses on CPU load prediction with uncertainty quantification using Bayesian linear regression. The resulting uncertainty estimates may support downstream tasks such as overload detection, anomaly monitoring, and risk-aware resource allocation.

In [ ]:
print('FINAL SUMMARY')
print('=' * 70)
print()
print('Metropolis-Hastings (MH):')
print(f'  + Simple to implement, no gradient required')
print(f'  - Random walk behavior leads to slow exploration')
print(f'  - Low ESS relative to number of samples')
print(f'  Acceptance Rate: {results["MH"]["acceptance_rate"]:.3f}')
print(f'  Avg ESS: {results["MH"]["avg_ess"]:.1f}')
print(f'  RMSE: {results["MH"]["predictions"]["rmse"]:.4f}')
print()
print('Gibbs Sampling:')
print(f'  + 100% acceptance rate (exact conditionals)')
print(f'  + Near-maximal ESS in this conjugate setting')
print(f'  - Requires known full conditional distributions')
print(f'  - Can be slow with strong correlations')
print(f'  Acceptance Rate: {results["Gibbs"]["acceptance_rate"]:.3f}')
print(f'  Avg ESS: {results["Gibbs"]["avg_ess"]:.1f}')
print(f'  RMSE: {results["Gibbs"]["predictions"]["rmse"]:.4f}')
print()
print('Hamiltonian Monte Carlo (HMC):')
print(f'  + Gradient-informed proposals, high ESS')
print(f'  + Explores parameter space efficiently')
print(f'  - Requires differentiable log-posterior')
print(f'  - Each iteration is computationally expensive (leapfrog)')
print(f'  Acceptance Rate: {results["HMC"]["acceptance_rate"]:.3f}')
print(f'    Note: The high acceptance rate (~95%) may indicate a')
print(f'    conservative step size; a larger step size could improve')
print(f'    ESS/sec at the cost of lower acceptance.')
print(f'  Avg ESS: {results["HMC"]["avg_ess"]:.1f}')
print(f'  RMSE: {results["HMC"]["predictions"]["rmse"]:.4f}')
print()
print('OLS Baseline RMSE: {:.4f}'.format(ols_rmse))
print('All three Bayesian methods achieve comparable RMSE to OLS but')
print('additionally provide conservative uncertainty estimates through')
print('posterior predictive intervals (coverage > nominal levels).')
print()
print('=' * 70)
print('All three methods approximate the same posterior distribution,')
print('providing evidence for implementation correctness.')
print('The key differences lie in efficiency (ESS/sec) and ease of use.')
print()
print('Limitations:')
print('- CPU Usage (MHz) was excluded from features to avoid data leakage.')
print('- All lag/rolling features use past values only.')
print('- The analysis uses the earliest 5,000 observations (sorted by time)')
print('  from 50 VMs, which may not represent the full range of workload')
print('  patterns across the complete 30-day trace.')